# Predicting WPPSI-III IQ from neonatal HRV — leakage-free regression pipeline

This notebook estimates how well a child's Full-Scale IQ (**QIT**) can be predicted from
heart-rate-variability (HRV) features measured at birth (recording period `M0`).

What it does, in order:
1. loads the three tab-separated source files and builds one modelling table per target,
2. learns all data-dependent preprocessing (correlation / variance filters, imputation,
   scaling, Lasso screening) **inside** every cross-validation fold,
3. compares ElasticNet, Random Forest, LightGBM, XGBoost and SVR against a mean-predictor
   baseline with repeated nested cross-validation,
4. plots the comparison, analyses the residuals of the best model, refits and stores the
   final models, runs an optional SHAP-based feature-count study and writes `SUMMARY.md`.

Run the notebook from the folder that holds `.env` and `pipeline_components.py`
(copy `env.example` to `.env` first). Research use only — not a diagnostic tool.
Health data and every file derived from individual children must stay local.

## 1. Setup

In [ ]:
import hashlib
import subprocess
import time
import os
from datetime import date
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import find_dotenv, load_dotenv
from joblib import Parallel, delayed
from matplotlib.ticker import MaxNLocator
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, KFold, RepeatedKFold

from pipeline_components import (
    BASELINE_LABEL,
    MODEL_STEP,
    build_baseline_spec,
    build_method_registry,
    build_preprocessor,
    fit_and_score_fold,
    make_search,
    regression_scores,
)

try:  # nicer tables inside Jupyter, plain print elsewhere
    from IPython.display import display
except ImportError:
    display = print

sns.set_style("whitegrid")

### Configuration

Every path and run option comes from the `.env` file (see `env.example`).

In [ ]:
load_dotenv(find_dotenv(usecwd=True), override=True)


def read_setting(name, default=None):
    """Return an environment setting as a clean string, failing loudly if it is absent."""
    value = os.getenv(name, default)
    if value is None or str(value).strip() == "":
        raise EnvironmentError(
            f"Setting '{name}' is missing. Copy env.example to .env and fill it in."
        )
    return str(value).strip()


# --- input files (tab separated, despite the .csv extension) ---
HRV_LINEAR_FILE = Path(read_setting("HRV_LINEAR_FILE"))
HRV_NONLINEAR_FILE = Path(read_setting("HRV_NONLINEAR_FILE"))
WPPSI_FILE = Path(read_setting("WPPSI_FILE"))

# --- output folders: the first four hold data derived from children, keep them out of git ---
PROCESSED_DIR = Path(read_setting("PROCESSED_DIR"))
MODELS_DIR = Path(read_setting("MODELS_DIR"))
PLOTS_DIR = Path(read_setting("PLOTS_DIR"))
CACHE_DIR = PROCESSED_DIR / "cv_cache"
# aggregated numbers only (no individual rows): safe to share
RESULTS_DIR = Path(read_setting("RESULTS_DIR"))
for folder in (PROCESSED_DIR, MODELS_DIR, PLOTS_DIR, CACHE_DIR, RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# --- evaluation protocol ---
SEED = int(read_setting("RANDOM_SEED", "42"))
OUTER_SPLITS = int(read_setting("OUTER_SPLITS", "5"))
OUTER_REPEATS = int(read_setting("OUTER_REPEATS", "10"))
INNER_SPLITS = int(read_setting("INNER_SPLITS", "5"))
SEARCH_ITERATIONS = int(read_setting("SEARCH_ITERATIONS", "50"))
N_JOBS = int(read_setting("N_JOBS", "-1"))
RESUME_RUNS = read_setting("RESUME_RUNS", "1") == "1"
RUN_SHAP = read_setting("RUN_SHAP", "1") == "1"
SHAP_MAX_FEATURES = int(read_setting("SHAP_MAX_FEATURES", "8"))

# --- targets: short name -> column label in the merged table ---
TARGET_COLUMNS = {
    "QIT": "Note composite_(QIT)",  # full-scale IQ
    "QIV": "Note composite_(QIV)",  # verbal IQ
    "QIP": "Note composite_(QIP)",  # performance IQ
    "NCL": "Note composite_(NCL)",  # non-verbal cognitive level
}
ACTIVE_TARGETS = [name.strip() for name in read_setting("TARGETS", "QIT").split(",")]
unknown = [name for name in ACTIVE_TARGETS if name not in TARGET_COLUMNS]
if unknown:
    raise ValueError(f"Unknown target(s) {unknown}; choose among {list(TARGET_COLUMNS)}")

print(f"targets: {ACTIVE_TARGETS} | outer CV: {OUTER_SPLITS} folds x {OUTER_REPEATS} repeats "
      f"= {OUTER_SPLITS * OUTER_REPEATS} fits per model | inner CV: {INNER_SPLITS} folds | "
      f"search budget: {SEARCH_ITERATIONS}")

## 2. Loading the source files

The HRV files start with two header lines (feature name, then unit). The unit line has no
entries for the three leading identifier columns, so it is shifted right by three.
The WPPSI file also has two header lines (group label, then variable name); group labels
are only written on the first column of each group and are therefore propagated rightwards.
Both headers are collapsed into one string per column: `name_(unit)` / `group_(name)`.

In [ ]:
ID_KEYS = ["Monogramme", "DDN"]  # child identifier and date of birth
PERIOD_COLUMN = "Passage_(-)"  # recording period of the HRV measurement
BIRTH_PERIOD = "M0"
ENCODING_FIXES = {"Num�ro": "Numéro", "Diab�te": "Diabète", "Pr�matur�": "Prématuré"}
MISSING_TOKENS = ["-", "", " ", "nan", "NaN"]


def join_label(name, qualifier):
    """Combine a name and its unit/description into a single column label."""
    qualifier = "" if pd.isna(qualifier) else str(qualifier)
    return f"{name}_({qualifier})" if qualifier else str(name)


def read_hrv_table(path):
    """Read one HRV export and return it with flattened column labels."""
    raw = pd.read_csv(path, sep="\t", header=None, encoding="utf-8-sig")
    names = raw.iloc[0].replace(ENCODING_FIXES).tolist()
    units = [""] * 3 + raw.iloc[1].fillna("").tolist()
    table = raw.iloc[2:].reset_index(drop=True)
    table.columns = [join_label(n, u) for n, u in zip(names, units[: len(names)])]
    return table


def read_wppsi_table(path):
    """Read the WPPSI-III export and return it with flattened column labels."""
    raw = pd.read_csv(path, sep="\t", header=None)
    groups = raw.iloc[0].ffill().fillna("")
    variables = raw.iloc[1]
    # the identifier block has no group label: column 1 is the child id, column 2 the birth date
    if not (pd.isna(variables.iloc[1]) and variables.iloc[2] == "DDN"):
        raise ValueError("Unexpected WPPSI header layout; check the input file.")
    labels = [f"{group}_({variable})" for group, variable in zip(groups, variables)]
    labels[1], labels[2] = "Monogramme", "DDN"
    table = raw.iloc[2:].reset_index(drop=True)
    table.columns = labels
    return table


linear_raw = read_hrv_table(HRV_LINEAR_FILE)
nonlinear_raw = read_hrv_table(HRV_NONLINEAR_FILE)
wppsi_raw = read_wppsi_table(WPPSI_FILE)

# only shapes are printed on purpose: individual rows should not end up in notebook outputs
print("linear HRV     :", linear_raw.shape)
print("non-linear HRV :", nonlinear_raw.shape)
print("WPPSI-III      :", wppsi_raw.shape)

## 3. Building the modelling table

Everything in this step is deterministic and independent of the target values, apart from
discarding children without a score:

* keep only birth (`M0`) recordings in each HRV file,
* keep WPPSI rows that have the requested score,
* inner-join the two HRV tables on id + birth date + period, then add the score,
* turn placeholder strings (`-`, `NaN`, blanks) into real missing values,
* drop empty columns and columns with more than 80 % missing values,
* convert to numbers and discard columns that are not numeric at all,
* remove identifiers and any other WPPSI composite so that no outcome information is left
  among the predictors.

Statistics that depend on the data (imputation means, correlations, variances, scaling,
Lasso screening) are **not** applied here; they live in the pipeline and are re-learned
in every training fold.

In [ ]:
MAX_MISSING_FRACTION = 0.80
IDENTIFIER_COLUMNS = {"Numéro_x", "Numéro_y"}


def select_period(table, period):
    """Rows of ``table`` recorded in the given period (whitespace-insensitive)."""
    in_period = table[PERIOD_COLUMN].astype("string").str.strip().eq(period)
    return table.loc[in_period.fillna(False).to_numpy(dtype=bool)].copy()


def assemble_dataset(target_key):
    """Return (features, target) for one IQ score as numeric pandas objects."""
    target_label = TARGET_COLUMNS[target_key]
    if target_label not in wppsi_raw.columns:
        raise KeyError(f"Column '{target_label}' not found in the WPPSI table.")

    linear_birth = select_period(linear_raw, BIRTH_PERIOD)
    nonlinear_birth = select_period(nonlinear_raw, BIRTH_PERIOD)
    scored = wppsi_raw.dropna(subset=[target_label])[[*ID_KEYS, target_label]]

    hrv = linear_birth.merge(nonlinear_birth, on=[*ID_KEYS, PERIOD_COLUMN], how="inner")
    merged = hrv.merge(scored, on=ID_KEYS, how="inner")

    merged = merged.replace(MISSING_TOKENS, np.nan).dropna(axis=1, how="all")
    too_sparse = merged.columns[(merged.isna().mean() > MAX_MISSING_FRACTION).to_numpy()]
    merged = merged.drop(columns=too_sparse)

    numeric = merged.apply(pd.to_numeric, errors="coerce")
    numeric = numeric.loc[:, numeric.notna().any()]  # text-only columns become all-NaN
    target = numeric.pop(target_label)

    discard = [
        column for column in numeric.columns
        if column in IDENTIFIER_COLUMNS or "Note composite" in column
    ]
    features = numeric.drop(columns=discard)

    duplicated_children = int(merged.duplicated(subset=ID_KEYS).sum())
    print(
        f"[{target_key}] children: {len(features)} | predictors: {features.shape[1]} | "
        f"sparse columns removed: {len(too_sparse)} | duplicated ids: {duplicated_children}"
    )
    return features.astype(float), target.astype(float)


modeling_tables = {key: assemble_dataset(key) for key in ACTIVE_TARGETS}

# Keep a local copy of each table (raw numbers, gaps left in place).
for key, (features, target) in modeling_tables.items():
    pd.concat([features, target], axis=1).to_csv(
        PROCESSED_DIR / f"modeling_table_{key}.csv", index=False
    )

## 4. Preprocessing inside the pipeline

`build_preprocessor()` chains five steps, all fitted on training data only:

1. correlation pruning (|r| > 0.95),
2. near-zero variance removal (variance < 0.01),
3. mean imputation,
4. `StandardScaler`,
5. Lasso screening (`alpha = 0.01`, keep non-zero coefficients).

The cell below fits the chain once on the whole table, purely to report how many
predictors each stage keeps. These numbers are descriptive; the evaluation never uses
this full-data fit.

In [ ]:
METHOD_REGISTRY = build_method_registry(seed=SEED)
BASELINE_SPEC = build_baseline_spec()
print("registered methods:", list(METHOD_REGISTRY))


def report_feature_funnel(features, target):
    """Number of predictors left after each preprocessing stage (full-data fit, descriptive)."""
    chain = build_preprocessor().fit(features, target)
    rows = [{"stage": "input", "n_features": features.shape[1]}]
    for position, (stage_name, _) in enumerate(chain.steps, start=1):
        rows.append(
            {"stage": stage_name, "n_features": chain[:position].transform(features).shape[1]}
        )
    return pd.DataFrame(rows), list(chain.get_feature_names_out())


for key, (features, target) in modeling_tables.items():
    funnel, kept = report_feature_funnel(features, target)
    print(f"\n[{key}] predictors surviving each stage")
    display(funnel)

## 5. Nested cross-validation

* **Outer loop** — `RepeatedKFold` produces the performance estimate. The splits are built
  once per target and shared by every model, so models are compared on identical folds.
* **Inner loop** — a shuffled `KFold` inside each outer training set selects the
  hyper-parameters by MAE (exhaustive grid when the space is small, random sampling otherwise).
* Every outer fit refits the whole pipeline (filters, scaler, Lasso, regressor) on the
  training rows alone.
* A model that always predicts the training mean is evaluated on the same folds as a reference.

Finished model/target runs are cached on disk, so an interrupted session can resume.

In [ ]:
evaluation_store = {}  # (target, model label) -> dict with fold table and out-of-fold predictions


def run_fingerprint(target_key, spec, features):
    """Short hash of everything that influences a run, used to name its cache file.

    Besides the protocol settings it covers the data layout, the preprocessing chain and the
    hyper-parameter space, so a stale cache is never reused after one of them changes.
    """
    description = "|".join(map(str, [
        target_key, spec.label, features.shape, "\x1f".join(features.columns),
        OUTER_SPLITS, OUTER_REPEATS, INNER_SPLITS, SEARCH_ITERATIONS, SEED,
        build_preprocessor(), spec.prefixed_space(),
    ]))
    return hashlib.md5(description.encode()).hexdigest()[:10]


def evaluate_method(target_key, spec, features, target, splits):
    """Nested-CV evaluation of one method on one target; returns fold scores and OOF predictions."""
    cache_file = CACHE_DIR / f"cv_{target_key}_{spec.label}_{run_fingerprint(target_key, spec, features)}.joblib"
    if RESUME_RUNS and cache_file.exists():
        print(f"  {spec.label}: loaded from cache")
        return joblib.load(cache_file)

    begin = time.perf_counter()
    outcomes = Parallel(n_jobs=N_JOBS)(
        delayed(fit_and_score_fold)(
            spec, features, target, train_idx, test_idx, INNER_SPLITS, SEARCH_ITERATIONS, SEED
        )
        for train_idx, test_idx in splits
    )

    rows = []
    out_of_fold = np.full((OUTER_REPEATS, len(features)), np.nan)
    for position, outcome in enumerate(outcomes):
        repeat, fold = divmod(position, OUTER_SPLITS)
        rows.append({
            "model": spec.label, "target": target_key, "repeat": repeat, "fold": fold,
            **outcome["scores"],
            "n_warnings": sum(outcome["warnings"].values()),
            "warning_types": ", ".join(sorted(outcome["warnings"])),
            "best_params": str(outcome["best_params"]),
            "seconds": outcome["seconds"],
        })
        out_of_fold[repeat, outcome["test_idx"]] = outcome["predictions"]

    record = {"folds": pd.DataFrame(rows), "oof": out_of_fold}
    joblib.dump(record, cache_file)
    print(f"  {spec.label}: done in {(time.perf_counter() - begin) / 60:.1f} min")
    return record


outer_cv = RepeatedKFold(n_splits=OUTER_SPLITS, n_repeats=OUTER_REPEATS, random_state=SEED)

for key, (features, target) in modeling_tables.items():
    print(f"\n=== target {key} ===")
    outer_splits = list(outer_cv.split(features))
    for spec in [*METHOD_REGISTRY.values(), BASELINE_SPEC]:
        evaluation_store[(key, spec.label)] = evaluate_method(
            key, spec, features, target, outer_splits
        )

## 6. Results table

Mean ± standard deviation over all outer folds × repeats (population SD).
`results/model_comparison.csv` follows the agreed layout; fold-level numbers go to
`fold_scores.csv`.

In [ ]:
fold_scores = pd.concat([rec["folds"] for rec in evaluation_store.values()], ignore_index=True)

METRICS = ["mae", "mse", "rmse", "r2"]
summary_rows = []
for (key, label), record in evaluation_store.items():
    row = {"model": label, "target": key}
    for metric in METRICS:
        values = record["folds"][metric].to_numpy()
        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_std"] = values.std()
    summary_rows.append(row)

comparison = pd.DataFrame(summary_rows)
comparison = comparison[
    ["model", "target"] + [f"{m}_{s}" for m in METRICS for s in ("mean", "std")]
]
comparison.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
fold_scores.drop(columns=["best_params"]).to_csv(RESULTS_DIR / "fold_scores.csv", index=False)
fold_scores[["model", "target", "repeat", "fold", "best_params"]].to_csv(
    PROCESSED_DIR / "fold_best_params.csv", index=False
)

display(comparison.round(3).sort_values(["target", "mae_mean"]))

## 7. Comparison plots

In [ ]:
MODEL_ORDER = [*METHOD_REGISTRY, BASELINE_LABEL]
palette = dict(zip(MODEL_ORDER, sns.color_palette("viridis", len(MODEL_ORDER))))
palette[BASELINE_LABEL] = "#9e9e9e"


def grouped_bars(metric, ylabel, title, filename, zero_line=False):
    """One cluster of bars per target, one bar per model, with SD error bars."""
    fig, ax = plt.subplots(figsize=(max(9, 3 * len(ACTIVE_TARGETS) + 5), 5.5))
    width = 0.8 / len(MODEL_ORDER)
    for offset, label in enumerate(MODEL_ORDER):
        subset = comparison[comparison["model"] == label].set_index("target").reindex(ACTIVE_TARGETS)
        positions = np.arange(len(ACTIVE_TARGETS)) + (offset - (len(MODEL_ORDER) - 1) / 2) * width
        bars = ax.bar(positions, subset[f"{metric}_mean"], width=width * 0.92,
                      yerr=subset[f"{metric}_std"], capsize=3, label=label, color=palette[label])
        ax.bar_label(bars, fmt="%.2f", label_type="edge", padding=2, fontsize=7)
    if zero_line:
        ax.axhline(0, color="red", linestyle="--", linewidth=1.2, label="R² = 0")
        low, high = ax.get_ylim()
        ax.set_ylim(low, max(high, 0) + 0.06 * (high - low))  # keep the zero line off the frame
    ax.set_xticks(np.arange(len(ACTIVE_TARGETS)))
    ax.set_xticklabels(ACTIVE_TARGETS)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


grouped_bars("mae", "MAE (IQ points)", "Mean absolute error by model", "model_comparison_mae.png")
grouped_bars("r2", "R²", "R² by model (dashed line: R² = 0)", "model_comparison_r2.png",
             zero_line=True)

for key in ACTIVE_TARGETS:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    subset = fold_scores[fold_scores["target"] == key]
    for ax, metric in zip(axes.ravel(), METRICS):
        sns.boxplot(data=subset, x="model", y=metric, order=MODEL_ORDER,
                    palette=palette, hue="model", legend=False, ax=ax)
        ax.set_title(f"{key} — {metric.upper()} per outer fold")
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=40)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"model_comparison_boxplots_{key}.png", dpi=300, bbox_inches="tight")
    plt.show()

## 8. Best model per target and sanity checks

* best by lowest mean MAE and by highest mean R² (baseline excluded), flagged when R² > 0,
* paired comparison with the baseline on identical folds,
* review of warnings, failures and fold-to-fold spread.

In [ ]:
ranked = comparison[comparison["model"] != BASELINE_LABEL].copy()
ranked["r2_above_zero"] = ranked["r2_mean"] > 0
best_by_mae = ranked.loc[ranked.groupby("target")["mae_mean"].idxmin()].set_index("target")
best_by_r2 = ranked.loc[ranked.groupby("target")["r2_mean"].idxmax()].set_index("target")

print("Lowest MAE per target:")
display(best_by_mae[["model", "mae_mean", "mae_std", "r2_mean", "r2_above_zero"]].round(3))
print("Highest R² per target:")
display(best_by_r2[["model", "r2_mean", "r2_std", "mae_mean"]].round(3))

for key in ACTIVE_TARGETS:
    if best_by_r2.loc[key, "r2_mean"] <= 0:
        print(f"[{key}] no model has a mean R² above zero: none beats a constant prediction in R² terms.")

# paired comparison against the baseline
fold_order = ["repeat", "fold"]
diagnostics = []
for key in ACTIVE_TARGETS:
    base_mae = (fold_scores[(fold_scores.target == key) & (fold_scores.model == BASELINE_LABEL)]
                .sort_values(fold_order)["mae"].to_numpy())
    for label in METHOD_REGISTRY:
        own = fold_scores[(fold_scores.target == key) & (fold_scores.model == label)].sort_values(fold_order)
        gain = base_mae - own["mae"].to_numpy()  # positive: the model is more accurate than the baseline
        diagnostics.append({
            "target": key, "model": label,
            "mae_gain_vs_baseline": gain.mean(),
            "folds_better_than_baseline": f"{(gain > 0).mean():.0%}",
            "total_warnings": int(own["n_warnings"].sum()),
            "mae_min": own["mae"].min(), "mae_max": own["mae"].max(),
        })
display(pd.DataFrame(diagnostics).round(3))

## 9. Error analysis of the best model

Each child is a test case once per repeat, so averaging the out-of-fold predictions over
repeats gives one honest prediction per child. The figure shows predicted vs. actual,
residuals vs. predicted and the residual distribution; the table lists the largest errors
by row position (no identifiers).

In [ ]:
def analyse_errors(target_key, model_label):
    """Residual diagnostics for ``model_label`` on ``target_key`` from out-of-fold predictions."""
    actual = modeling_tables[target_key][1].to_numpy()
    predicted = np.nanmean(evaluation_store[(target_key, model_label)]["oof"], axis=0)
    residual = actual - predicted

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    axes[0].scatter(predicted, actual, alpha=0.6)
    bounds = [min(actual.min(), predicted.min()), max(actual.max(), predicted.max())]
    axes[0].plot(bounds, bounds, color="red", linestyle="--")
    axes[0].set(xlabel="Predicted", ylabel="Actual", title="Predicted vs actual")

    axes[1].scatter(predicted, residual, alpha=0.6)
    axes[1].axhline(0, color="red", linestyle="--")
    axes[1].set(xlabel="Predicted", ylabel="Residual (actual − predicted)", title="Residuals vs predicted")

    sns.histplot(residual, kde=True, ax=axes[2])
    axes[2].set(xlabel="Residual", title="Residual distribution")

    fig.suptitle(f"{model_label} — target {target_key}")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"error_analysis_{model_label}_{target_key}.png", dpi=300, bbox_inches="tight")
    plt.show()

    spread = residual.std()
    print(f"residual mean {residual.mean():.2f}, SD {spread:.2f}; "
          f"{int((np.abs(residual) > 2 * spread).sum())} children with |residual| > 2 SD")
    worst = (pd.DataFrame({"row": np.arange(len(actual)), "actual": actual,
                           "predicted": predicted, "residual": residual})
             .assign(abs_error=lambda t: t["residual"].abs())
             .sort_values("abs_error", ascending=False).head(10))
    display(worst.round(2))


for key in ACTIVE_TARGETS:
    analyse_errors(key, best_by_mae.loc[key, "model"])

## 10. Final models

Each method is tuned once more on **all** children and stored with joblib. The stored object
is the complete pipeline (filters → imputer → scaler → Lasso → regressor), so it can be
applied to new raw feature tables directly; the fitted scaler is also saved on its own.
These refits are for reuse and interpretation, not for performance estimation.

In [ ]:
final_models = {}
selected_feature_names = {}
final_inner_cv = KFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED)

for key, (features, target) in modeling_tables.items():
    for label, spec in METHOD_REGISTRY.items():
        search = make_search(spec, final_inner_cv, SEARCH_ITERATIONS, SEED)
        search.fit(features, target)
        final_models[(key, label)] = search.best_estimator_
        joblib.dump(search.best_estimator_, MODELS_DIR / f"best_{label.lower()}_{key}.joblib")
        print(f"[{key}] {label}: inner-CV MAE {-search.best_score_:.3f} | {search.best_params_}")

    reference_pipeline = final_models[(key, next(iter(METHOD_REGISTRY)))]
    joblib.dump(reference_pipeline.named_steps["scaler"], MODELS_DIR / f"scaler_{key}.joblib")
    selected_feature_names[key] = list(reference_pipeline[:-1].get_feature_names_out())
    pd.Series(selected_feature_names[key], name="feature").to_csv(
        RESULTS_DIR / f"selected_features_{key}.csv", index=False
    )

## 11. SHAP-based feature-count study (exploratory)

For each method: inside every training fold the preprocessing is refitted, a model is
trained on the retained predictors and predictors are ranked by mean |SHAP| **computed on
the training rows only**. Models using the top 1…K predictors are then scored on the
untouched test rows. Tree models use `TreeExplainer`; SVR uses a model-agnostic explainer.
ElasticNet borrows the Random Forest ranking (its Lasso-screened coefficients are often all
shrunk to zero) and is re-tuned for every feature count.

Hyper-parameters of the ranking/evaluation models come from the final refit in section 10,
which saw all children, so the numbers remain exploratory; the nested CV above is the
reference performance estimate.

In [ ]:
if RUN_SHAP:
    try:
        import shap
    except ImportError:
        shap = None
        print("shap is not installed: skipping the SHAP study.")
else:
    shap = None
    print("RUN_SHAP=0: skipping the SHAP study.")


def mean_abs_shap(fitted_model, matrix, tree_based):
    """Average absolute SHAP value of each column of ``matrix``."""
    if tree_based:
        values = shap.TreeExplainer(fitted_model).shap_values(matrix)
    else:
        background = shap.maskers.Independent(matrix, max_samples=100)
        values = shap.Explainer(fitted_model.predict, background)(matrix).values
    values = np.asarray(values)
    if values.ndim == 3:  # multi-output layout: collapse the last axis
        values = np.abs(values).mean(axis=2)
    return np.abs(values).mean(axis=0)


def feature_count_study(target_key, method_label):
    """Fold-wise scores of ``method_label`` when restricted to its top-k SHAP-ranked predictors."""
    features, target = modeling_tables[target_key]
    spec = METHOD_REGISTRY[method_label]
    ranker_label = "RandomForest" if method_label == "ElasticNet" else method_label
    ranker_template = final_models[(target_key, ranker_label)].named_steps[MODEL_STEP]
    scored_template = final_models[(target_key, method_label)].named_steps[MODEL_STEP]
    tree_based = METHOD_REGISTRY[ranker_label].tree_based

    splitter = KFold(n_splits=5, shuffle=True, random_state=SEED)
    score_rows, importance_rows = [], []
    for fold_no, (train_idx, test_idx) in enumerate(splitter.split(features), start=1):
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]
        prep = build_preprocessor()
        Z_train = prep.fit_transform(features.iloc[train_idx], y_train)
        Z_test = prep.transform(features.iloc[test_idx])
        names = prep.get_feature_names_out()

        ranker = clone(ranker_template).fit(Z_train, y_train)
        importance = mean_abs_shap(ranker, Z_train, tree_based)
        order = np.argsort(importance)[::-1]
        importance_rows += [{"fold": fold_no, "feature": n, "importance": v}
                            for n, v in zip(names, importance)]

        for k in range(1, min(SHAP_MAX_FEATURES, len(order)) + 1):
            columns = order[:k]
            if method_label == "ElasticNet":
                inner = GridSearchCV(
                    clone(scored_template), spec.search_space,
                    scoring="neg_mean_absolute_error",
                    cv=KFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED), n_jobs=1,
                )
                fitted = inner.fit(Z_train[:, columns], y_train).best_estimator_
            else:
                fitted = clone(scored_template).fit(Z_train[:, columns], y_train)
            score_rows.append({
                "model": method_label, "target": target_key, "n_features": k, "fold": fold_no,
                **regression_scores(y_test, fitted.predict(Z_test[:, columns])),
            })
    return pd.DataFrame(score_rows), pd.DataFrame(importance_rows)


shap_tables, shap_best = [], []
if shap is not None:
    for key in ACTIVE_TARGETS:
        for label in METHOD_REGISTRY:
            fold_level, importance_long = feature_count_study(key, label)
            shap_tables.append(fold_level)
            # ranking averaged over folds; a predictor absent from a fold counts as importance 0
            ranking = (importance_long.pivot_table(index="feature", columns="fold", values="importance")
                       .fillna(0).mean(axis=1).sort_values(ascending=False).rename("mean_abs_shap"))
            ranking.to_csv(RESULTS_DIR / f"shap_ranking_{label}_{key}.csv")
            print(f"[{key}] {label}: study finished")

    shap_scores = pd.concat(shap_tables, ignore_index=True)
    shap_summary = (shap_scores.groupby(["target", "model", "n_features"])[["mae", "rmse", "r2"]]
                    .agg(["mean", "std"]))
    shap_summary.columns = ["_".join(col) for col in shap_summary.columns]
    shap_summary = shap_summary.reset_index()
    shap_summary.to_csv(RESULTS_DIR / "shap_feature_count_results.csv", index=False)

    shap_best = (shap_summary.loc[shap_summary.groupby(["target", "model"])["mae_mean"].idxmin()]
                 .sort_values(["target", "mae_mean"]).reset_index(drop=True))
    shap_best.to_csv(RESULTS_DIR / "shap_best_feature_count.csv", index=False)
    display(shap_best.round(3))

    for key in ACTIVE_TARGETS:
        fig, ax = plt.subplots(figsize=(9, 5.5))
        sns.lineplot(data=shap_summary[shap_summary.target == key], x="n_features", y="mae_mean",
                     hue="model", marker="o", ax=ax)
        reference = comparison[(comparison.target == key) & (comparison.model == BASELINE_LABEL)]
        ax.axhline(reference["mae_mean"].iloc[0], color="grey", linestyle="--", label="mean baseline")
        ax.set(xlabel="Number of top-ranked predictors", ylabel="MAE", title=f"{key}: MAE vs predictor count")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
        fig.tight_layout()
        fig.savefig(PLOTS_DIR / f"shap_feature_count_mae_{key}.png", dpi=300, bbox_inches="tight")
        plt.show()

## 12. Written summary

`results/SUMMARY.md` is generated from the numbers above: best model per target, how it
compares with the baseline, limitations and next steps.

In [ ]:
def markdown_table(frame):
    """Render a small DataFrame as a GitHub-flavoured markdown table."""
    header = "| " + " | ".join(frame.columns) + " |"
    divider = "|" + "|".join("---" for _ in frame.columns) + "|"
    body = ["| " + " | ".join(str(v) for v in row) + " |" for row in frame.itertuples(index=False)]
    return "\n".join([header, divider, *body])


summary_lines = [
    "# Results summary",
    "",
    f"Generated automatically on {date.today().isoformat()}.",
    f"Protocol: nested cross-validation, outer {OUTER_SPLITS}-fold × {OUTER_REPEATS} repeats, "
    f"inner {INNER_SPLITS}-fold, search budget {SEARCH_ITERATIONS}, seed {SEED}. "
    "All preprocessing is fitted inside each training fold.",
    "",
    "## Best model per target",
    "",
]
best_rows = []
for key in ACTIVE_TARGETS:
    top = best_by_mae.loc[key]
    base = comparison[(comparison.target == key) & (comparison.model == BASELINE_LABEL)].iloc[0]
    best_rows.append({
        "target": key, "model": top["model"],
        "MAE": f"{top['mae_mean']:.2f} ± {top['mae_std']:.2f}",
        "RMSE": f"{top['rmse_mean']:.2f} ± {top['rmse_std']:.2f}",
        "R²": f"{top['r2_mean']:.3f} ± {top['r2_std']:.3f}",
        "baseline MAE": f"{base['mae_mean']:.2f}",
        "baseline R²": f"{base['r2_mean']:.3f}",
    })
summary_lines += [markdown_table(pd.DataFrame(best_rows)), ""]

for key in ACTIVE_TARGETS:
    if best_by_r2.loc[key, "r2_mean"] > 0:
        summary_lines.append(
            f"- **{key}**: the best mean R² is {best_by_r2.loc[key, 'r2_mean']:.3f} "
            f"({best_by_r2.loc[key, 'model']}); positive but to be read together with its spread."
        )
    else:
        summary_lines.append(
            f"- **{key}**: no model reaches a mean R² above zero, i.e. none explains more variance "
            "than always predicting the training mean."
        )

n_children, n_predictors = modeling_tables[ACTIVE_TARGETS[0]][0].shape
n_selected = len(selected_feature_names[ACTIVE_TARGETS[0]])
summary_lines += [
    "",
    "## Limitations",
    "",
    f"- Small sample: {n_children} children, {n_predictors} candidate predictors "
    f"({n_selected} kept by the Lasso screen when fitted on every child).",
    "- Single cohort: all estimates come from cross-validation on the same recruitment; there is no external validation.",
    "- Fold-to-fold spread is large relative to differences between models, so rankings are not decisive.",
    "- Mean imputation and linear screening may discard non-linear or missingness-related signal.",
    "",
    "## Next steps",
    "",
    "- Elastic Net with a Bayesian treatment of the hyper-parameters / coefficients.",
    "- Add clinical covariates beyond the HRV summary features.",
    "- Work from the raw ECG signal instead of pre-computed HRV indices.",
    "- Extend the evaluation to QIV, QIP and NCL (set `TARGETS` in `.env`).",
    "",
]
(RESULTS_DIR / "SUMMARY.md").write_text("\n".join(summary_lines), encoding="utf-8")
print((RESULTS_DIR / "SUMMARY.md").read_text(encoding="utf-8"))

## 13. Repository hygiene check

Lists any file tracked by git that looks like data or a trained model and lies outside the
shareable results folder.

In [ ]:
def audit_tracked_files():
    """Warn about tracked data/model files; silently skip when not inside a git repository."""
    try:
        tracked = subprocess.run(
            ["git", "ls-files"], capture_output=True, text=True, check=True
        ).stdout.splitlines()
        top_level = Path(subprocess.run(
            ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True, check=True
        ).stdout.strip())
    except (OSError, subprocess.CalledProcessError):
        print("Not inside a git repository (or git is unavailable): nothing to check.")
        return
    try:
        allowed_prefix = RESULTS_DIR.resolve().relative_to(top_level.resolve()).as_posix()
    except ValueError:
        allowed_prefix = None  # results folder lies outside the repository
    risky_suffixes = (".csv", ".tsv", ".xlsx", ".parquet", ".pkl", ".joblib", ".npy")
    flagged = [
        path for path in tracked
        if path.lower().endswith(risky_suffixes)
        and not (allowed_prefix and path.startswith(allowed_prefix + "/"))
    ]
    if flagged:
        print("WARNING - tracked files that may contain health data or models:")
        for path in flagged:
            print("  ", path)
    else:
        print("OK: no data or model files are tracked outside the results folder.")


audit_tracked_files()